# B2-020-language-transformers — Practice p18 — Solution

**Type:** integrative · **Difficulty:** advanced · **Concepts:** language-transformer, causal-language-modeling

*65 minutes.*  
**Set:** C  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260812`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:C6-pytorch`, `book1:C7-cnn-transfer`, `book1:C11-neural-training`, `B2-019-attention-transformers`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C7-cnn-transfer](../../../../book1/units/C7-cnn-transfer/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-019-attention-transformers](../../B2-019-attention-transformers/lesson.ipynb).

## Pinned reproducibility protocol

Use only `data/language_fixture.py`, seed `20260812`, and the literal causal train/held-out IDs.  Initialize the stated width-8 one-block causal model and distinct `Linear(8,12)` head from that seed.  AdamW uses `lr=0.03`, `weight_decay=0`, betas `(0.9,0.999)`, eps `1e-8`; run exactly 80 full-batch updates in stored ascending order with no shuffle, dropout, clipping, scheduler, or accumulation.

## Solution

The shifted target at each position is the next token. A single learned-position, pre-norm causal block is trained only on literal rows; the held-out rows are disjoint and evaluated without updates.

In [ ]:
import importlib.util

def load_literal_module(name, relative_path):
    spec = importlib.util.spec_from_file_location(name, relative_path)
    assert spec is not None and spec.loader is not None
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)

def sinusoidal_positions(length, width, device):
    positions = torch.arange(length, dtype=torch.float32, device=device).unsqueeze(1)
    dimensions = torch.arange(0, width, 2, dtype=torch.float32, device=device)
    angles = positions / (10000.0 ** (dimensions / width))
    table = torch.zeros(length, width, dtype=torch.float32, device=device)
    table[:, 0::2] = torch.sin(angles)
    table[:, 1::2] = torch.cos(angles)
    return table

class TinyEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(12, 8, padding_idx=0)
        self.norm1 = nn.LayerNorm(8, eps=1e-5)
        self.attention = nn.MultiheadAttention(8, 2, dropout=0.0, batch_first=True)
        self.norm2 = nn.LayerNorm(8, eps=1e-5)
        self.ff1 = nn.Linear(8, 16)
        self.ff2 = nn.Linear(16, 8)

    def forward(self, token_ids, *, mask_mode):
        length = token_ids.shape[1]
        positions = sinusoidal_positions(length, 8, token_ids.device)
        x = self.token_embedding(token_ids) + positions.unsqueeze(0)
        normalized = self.norm1(x)
        attention_mask = None
        if mask_mode == "causal":
            attention_mask = torch.triu(
                torch.ones(length, length, dtype=torch.bool, device=token_ids.device),
                diagonal=1,
            )
        elif mask_mode != "bidirectional":
            raise ValueError(f"unknown mask mode: {mask_mode}")
        attended, _ = self.attention(
            normalized,
            normalized,
            normalized,
            attn_mask=attention_mask,
            key_padding_mask=token_ids.eq(0),
            need_weights=False,
        )
        x = x + attended
        return x + self.ff2(F.gelu(self.ff1(self.norm2(x))))

fixture = load_literal_module("language_fixture_p18", "../data/language_fixture.py")

def shift_targets(tokens):
    return tokens[:, :-1], tokens[:, 1:]

def causal_loss(encoder, head, token_rows):
    inputs, targets = shift_targets(token_rows)
    logits = head(encoder(inputs, mask_mode="causal"))
    valid = targets.ne(0)
    return F.cross_entropy(logits[valid], targets[valid], reduction="mean")

torch.manual_seed(20260812)
encoder = TinyEncoder()
head = nn.Linear(8, 12)
train_rows = torch.tensor(fixture.CAUSAL_TRAIN_IDS, dtype=torch.int64)
heldout_rows = torch.tensor(fixture.CAUSAL_HELDOUT_IDS, dtype=torch.int64)
train_inputs, train_targets = shift_targets(train_rows)
initial_train_loss = causal_loss(encoder, head, train_rows).detach()
initial_heldout_loss = causal_loss(encoder, head, heldout_rows).detach()
optimizer = torch.optim.AdamW(
    [*encoder.parameters(), *head.parameters()], lr=0.03, weight_decay=0,
    betas=(0.9, 0.999), eps=1e-8,
)
for _ in range(80):
    optimizer.zero_grad(set_to_none=True)
    loss = causal_loss(encoder, head, train_rows)
    loss.backward()
    optimizer.step()
final_train_loss = causal_loss(encoder, head, train_rows).detach()
final_heldout_loss = causal_loss(encoder, head, heldout_rows).detach()
prefix_rows = heldout_rows[:, :-1].clone()
mutated_rows = prefix_rows.clone(); mutated_rows[:, 5] = torch.tensor([9, 4])
prefix_logits = head(encoder(prefix_rows, mask_mode="causal"))
mutated_logits = head(encoder(mutated_rows, mask_mode="causal"))

### Answer check

In [ ]:
assert train_rows.dtype == torch.int64
assert next(encoder.parameters()).dtype == head.weight.dtype == torch.float32
assert torch.equal(train_inputs, train_rows[:, :-1])
assert torch.equal(train_targets, train_rows[:, 1:])
assert train_targets[0, 0].item() == 4 and train_targets[0, -1].item() == 0
assert initial_train_loss.item() > final_train_loss.item()
assert initial_heldout_loss.item() > final_heldout_loss.item()
assert set(map(tuple, fixture.CAUSAL_TRAIN_IDS)).isdisjoint(map(tuple, fixture.CAUSAL_HELDOUT_IDS))
assert prefix_logits.shape == (2, 7, 12)
assert torch.allclose(prefix_logits[:, :5], mutated_logits[:, :5], atol=1e-5, rtol=1e-5)